# Compact Mamba: participant-disjoint gait classification

This experiment evaluates an 18-channel, width-128, two-block **unidirectional** Mamba classifier with residual connections, three LayerNorm layers, temporal max pooling and a two-logit head. There are no positional embeddings, additional MLP blocks, oversampling or augmentation.

## Evaluation contract
- Audit and deduplicate original recordings before constructing participant folds. All recordings/windows from one study-qualified identity remain together.
- **10 stratified outer test folds**, each with a separate inner training/validation split. Model selection sees only inner validation. This uses an inner holdout, not full inner cross-validation.
- Fit per-channel statistics and inverse-frequency loss weights on inner training only. Validation/test are not rebalanced.
- Select epochs and optimizer settings using **inner participant log loss**. Evaluate the chosen checkpoint on its outer test fold once, with no outer-development refit.
- Average probabilities within recordings, then average recordings equally within each participant. Use a fixed PD threshold of 0.5 (ties are PD).
- Report participant-level out-of-fold metrics and bootstrap 95% intervals, separately for every training seed. Window metrics are secondary.

Enable a Kaggle GPU and Internet for setup; keep the dataset attached. Results and plots stay in RAM. Optional export is at the end. Finish the prespecified experiment before inspecting outer results; do not use those scores for iterative tuning.

In [ ]:
# Fetch immutable source only when a local repository is not available.
from pathlib import Path
import hashlib
import subprocess
import sys
import tempfile

SOURCE_REV = '90ab51995b40842f3ced1886cb69c429bd7ffe20'
REPO_URL = 'https://github.com/AlexM3020/Subject-Independent-Evaluation-of-Mamba-for-Parkinson-s-Disease-Detection-from-Gait-Signals.git'
INSTALL_MAMBA = False  # Set True on Kaggle if mamba-ssm is not installed.

search_roots = [Path.cwd(), *Path.cwd().parents]
choices = [p for base in search_roots for p in (base, base / 'paper_repository')]
REPO_ROOT = next((p for p in choices if (p / 'gait/cross_validation.py').is_file()), None)
if REPO_ROOT is None:
    REPO_ROOT = Path(tempfile.gettempdir()) / ('compact-mamba-' + SOURCE_REV[:12])
    if not REPO_ROOT.exists():
        subprocess.run(['git', 'clone', '--no-checkout', REPO_URL, str(REPO_ROOT)], check=True)
    revision = subprocess.run(['git', '-C', str(REPO_ROOT), 'rev-parse', '--verify', SOURCE_REV], capture_output=True, text=True)
    if revision.returncode != 0:
        subprocess.run(['git', '-C', str(REPO_ROOT), 'fetch', 'origin', SOURCE_REV], check=True)
    subprocess.run(['git', '-C', str(REPO_ROOT), 'checkout', '--detach', SOURCE_REV], check=True)
REPO_ROOT = REPO_ROOT.resolve()
# Refuse a silently changed implementation, including stale local checkouts.
for relative in ('gait/cross_validation.py', 'gait/models.py'):
    expected = subprocess.run(['git', '-C', str(REPO_ROOT), 'show', SOURCE_REV + ':' + relative], capture_output=True, check=True).stdout
    actual = (REPO_ROOT / relative).read_bytes().replace(b'\r\n', b'\n')
    assert actual == expected.replace(b'\r\n', b'\n'), f'Source differs from pinned revision: {relative}'
sys.path.insert(0, str(REPO_ROOT))
if INSTALL_MAMBA:
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'mamba-ssm>=2.2,<3', '--no-build-isolation'], check=True)
print('Source revision:', SOURCE_REV)
print('Repository:', REPO_ROOT)
print('Dependencies: CUDA PyTorch, NumPy, scikit-learn, Matplotlib and mamba-ssm. Restart the kernel after installation if required.')

## Configuration and a small parameter search

The starting point uses 2,000 timesteps (20 seconds), nonoverlapping windows, batch size 32, Adam at 5e-5, 60 maximum epochs, stopping patience 10 and plateau-scheduler patience 5. Width/state/convolution/expansion are fixed at 128/32/32/2.

Only three optimizer candidates are searched: the reference rate, a lower rate, and mild L2 regularization. The smallest budget uses only the reference candidate. Declare that choice before examining test predictions.

One seed means **30 candidate fits**; three seeds mean **90 fits**. Start with one complete 10-fold run. To assess training randomness, set `TRAINING_SEEDS = (42, 123, 2026)` and rerun the same protocol. Fold assignments stay fixed across these seeds. A future context-length comparison (1,000 versus 2,000) is a separate planned experiment, not a reason to keep tuning against outer results.

The default has no augmentation and no oversampling. Loss weights balance training window classes; longer/more recordings still contribute more training windows per participant.

In [ ]:
import copy
import json
from collections import Counter
from dataclasses import asdict
import numpy as np
import torch
import matplotlib.pyplot as plt
from gait import cross_validation as cv
from gait.models import build_model
assert Path(cv.__file__).resolve().is_relative_to(REPO_ROOT), 'Restart kernel: gait was imported from another checkout'

# Preserve any completed-fold progress if this cell is accidentally rerun.
results = globals().get('results', {})
DATA_DIR = Path('/kaggle/input/datasets/alexmartinelliita/gait-in-parkinsons-disease/')
TRAINING_SEEDS = (42,)  # Confirmatory option: (42, 123, 2026)
CFG = cv.Protocol(
    seq_len=2000, overlap=0.0, outer_folds=10, inner_fraction=0.2,
    split_seed=42, training_seeds=TRAINING_SEEDS, batch_size=32,
    max_epochs=60, patience=10, scheduler_patience=5,
    scheduler_factor=0.5, min_lr=1e-7,
    bootstrap_samples=2000, bootstrap_seed=7301, num_workers=0,
)
CANDIDATES = [
    {'name': 'reference', 'lr': 5e-5, 'weight_decay': 0.0},
    {'name': 'lower_lr', 'lr': 2.5e-5, 'weight_decay': 0.0},
    {'name': 'mild_l2', 'lr': 5e-5, 'weight_decay': 1e-4},
]
# CANDIDATES = CANDIDATES[:1]  # Optional smaller prespecified baseline: 10 fits/seed.
RUN_EXPERIMENT = False  # Set True only after the data audit and GPU check succeed.
cv.validate_protocol(CFG, CANDIDATES)
PLAN = {'protocol': asdict(CFG), 'candidates': copy.deepcopy(CANDIDATES), 'source_revision': SOURCE_REV}
print(json.dumps(PLAN, indent=2))
print('Candidate fits:', CFG.outer_folds * len(TRAINING_SEEDS) * len(CANDIDATES))
print(cv.runtime_info())

## Original-data audit and fold assignments
Use original 19-column 100-Hz recordings, not previously oversampled or segmented exports. Exact numeric-force duplicates within an identity are excluded and logged, even if timestamps or whitespace differ. Identical signals assigned to different identities fail closed. Unsupported prepared filenames, invalid time axes and missing/nonfinite values also fail rather than being silently repaired. Exact hashes do not detect every transformed near-duplicate: original provenance still matters.

Recordings shorter than one complete window are explicitly excluded; incomplete tails are discarded from model inputs. Normalization uses every timestep of each eligible **inner-training recording once**, including its tail; validation and test never contribute to these statistics.

Counts below distinguish participants, recordings and windows. They need not have the same class proportions. No post-split balancing changes validation or test support.

In [ ]:
records, audit = cv.load_originals(DATA_DIR, seq_len=CFG.seq_len)
windows = cv.make_windows(records, CFG.seq_len, CFG.overlap)
splits = cv.make_splits(records, CFG)
print('File audit:', dict(Counter(r['status'] for r in audit)))
for r in audit:
    if r['status'] != 'included':
        print(r['status'], r['path'], r.get('duplicate_of', ''))
subject_labels = {r['subject']: r['label'] for r in records}
for label, name in ((0, 'Control'), (1, 'PD')):
    print(name, '| participants:', sum(y == label for y in subject_labels.values()),
          '| recordings:', sum(r['label'] == label for r in records),
          '| windows:', sum(w['label'] == label for w in windows))
for s in splits:
    print('Fold', s['fold'], {part: dict(Counter(subject_labels[p] for p in s[part])) for part in ('train', 'validation', 'test')})
# Label-free byte/force checks plus full participant assignments remain in RAM for export.
audit_fingerprint = hashlib.sha256(json.dumps(audit, sort_keys=True).encode()).hexdigest()
FROZEN_PLAN = copy.deepcopy(PLAN)
print('Audited dataset fingerprint:', audit_fingerprint)
print('All 10 outer folds assign each eligible participant to testing exactly once.')

## Real CUDA forward/backward check
This checks the actual Mamba selective-scan backend with convolution width 32. The wide convolution uses PyTorch depthwise convolution, bypassing the optional small-kernel causal-conv1d path. No custom global initialization overwrites Mamba's time-step parameters. A CPU substitute is never used for research results.

The check uses synthetic input only and reports no classification score. If it fails, fix the CUDA/backend installation before starting the experiment.

In [ ]:
GPU_READY = False
if not torch.cuda.is_available():
    print('CUDA is unavailable here. Data auditing is possible, but real training requires a Kaggle GPU.')
else:
    device = cv.require_cuda()
    cv.seed_everything(12345)
    smoke_model = build_model(**cv.ARCHITECTURE).to(device)
    parameter_count = sum(p.numel() for p in smoke_model.parameters())
    assert parameter_count == 275330, f'Unexpected architecture size: {parameter_count}'
    smoke_x = torch.randn(2, CFG.seq_len, 18, device=device)
    smoke_logits = smoke_model(smoke_x)
    assert smoke_logits.shape == (2, 2) and torch.isfinite(smoke_logits).all()
    torch.nn.functional.cross_entropy(smoke_logits, torch.tensor([0, 1], device=device)).backward()
    assert all(p.grad is not None and torch.isfinite(p.grad).all() for p in smoke_model.parameters())
    print('Real width-32 forward/backward passed. Parameters:', parameter_count)
    del smoke_model, smoke_x, smoke_logits
    torch.cuda.empty_cache()
    GPU_READY = True

## Run the fixed experiment
Only inner-validation participant log loss drives the plateau scheduler, early stopping, checkpoint choice and candidate choice. Lower is better; ties keep the first candidate. Training CE is class-weighted at window level, whereas validation NLL is unweighted at participant level, so their absolute values are not directly comparable.

The selected inner checkpoint is evaluated without refitting. Every candidate receives the same fold-specific initialization/shuffle seed for a fair optimizer comparison. Results are retained in `results`, including per-epoch histories, trial scores, selected settings, normalization parameters and aligned outer predictions. No outer performance is printed until selection across all folds/seeds finishes.

Do not rerun the configuration cell midway through an experiment. RAM progress survives an interrupted call in the current kernel, but not a Kaggle session reset. There is no automatic resume or checkpoint download; the optional export cell can preserve completed-fold outputs explicitly. Weights are released after scoring to limit memory.

In [ ]:
if RUN_EXPERIMENT:
    assert GPU_READY, 'Run and pass the real GPU check first'
    assert FROZEN_PLAN == {'protocol': asdict(CFG), 'candidates': CANDIDATES, 'source_revision': SOURCE_REV}, 'Configuration changed after audit; rerun audit before starting'
    if 'results' in globals() and results.get('runs'):
        raise RuntimeError('Existing results are in RAM. Export if desired, then explicitly delete results before restarting; no automatic overwrite.')
    results = {}
    try:
        cv.run_nested_cv(records, audit, CFG, CANDIDATES, progress=results, verbose=True)
    finally:
        results['source_revision'] = SOURCE_REV
        results['audited_dataset_fingerprint'] = audit_fingerprint
    print('Completed the entire fixed experiment. Outer results may now be inspected.')
else:
    print('Training disabled. Set RUN_EXPERIMENT = True after reviewing the audit and GPU check.')

## Participant-level out-of-fold results and uncertainty
Pooled metrics count each eligible participant once per training seed, unlike an unweighted mean of unequal-sized fold accuracies. Bootstrap intervals resample participants **within each class**, keeping natural class counts fixed, and condition on the fitted OOF predictions. They do not include uncertainty from refitting or different splits. Fold standard deviation is descriptive, not a confidence interval.

With multiple seeds, report each seed and the mean/standard deviation across seeds. Do not choose the best seed or concatenate repeated participant predictions as independent observations. All seeds use identical fold assignments. Small cohorts can produce wide or degenerate empirical bootstrap intervals, especially for perfect class-specific scores.

In [ ]:
if 'results' not in globals() or not results.get('complete'):
    print('No complete experiment yet; partial folds are not reported as complete cross-validation.')
else:
    for seed, summary in results['summaries'].items():
        print('\nTRAINING SEED', seed, '| unique participants:', summary['participant_metrics']['n'])
        for metric, ci in summary['confidence_intervals'].items():
            print(f"  {metric:20s} {ci['estimate']:.4f}  95% CI [{ci['lower']:.4f}, {ci['upper']:.4f}]")
        print('  Fold accuracy SD (not CI):', summary['fold_accuracy_std'])
        print('  Secondary pooled window metrics:', summary['window_metrics'])
        chosen = [r['trials'][r['selected_candidate']]['candidate']['name'] for r in results['runs'] if str(r['seed']) == seed]
        print('  Inner-selected candidates:', dict(Counter(chosen)))
    if len(results['summaries']) > 1:
        print('\nAcross-training-seed mean ± sample SD (same fold assignments):')
        for metric in ('accuracy', 'balanced_accuracy', 'macro_f1', 'sensitivity', 'specificity', 'roc_auc'):
            values = np.array([s['participant_metrics'][metric] for s in results['summaries'].values()])
            print(f'{metric:20s} {values.mean():.4f} ± {values.std(ddof=1):.4f}')

In [ ]:
# Display-only plots; no automatic saving or last-epoch padding.
if 'results' in globals() and results.get('complete'):
    from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix, roc_curve, roc_auc_score
    for seed in results['protocol']['training_seeds']:
        runs = [r for r in results['runs'] if r['seed'] == seed]
        participant_rows = [p for r in runs for p in r['test_participants']]
        y = np.array([p['label'] for p in participant_rows])
        probability = np.array([p['probability'] for p in participant_rows])
        fig, axes = plt.subplots(1, 2, figsize=(10, 4))
        ConfusionMatrixDisplay(confusion_matrix(y, probability >= 0.5, labels=[0, 1]), display_labels=['Control', 'PD']).plot(ax=axes[0], colorbar=False)
        axes[0].set_title(f'Participant OOF confusion matrix | seed {seed}')
        fpr, tpr, _ = roc_curve(y, probability)
        axes[1].plot(fpr, tpr, label=f'AUC = {roc_auc_score(y, probability):.3f}')
        axes[1].plot([0, 1], [0, 1], '--', color='gray')
        axes[1].set(xlabel='False positive rate', ylabel='True positive rate', title='Participant OOF ROC')
        axes[1].legend()
        fig.tight_layout()
        plt.show()
        fig, axes = plt.subplots(1, 2, figsize=(12, 4))
        for r in runs:
            trial = r['trials'][r['selected_candidate']]
            history = trial['history']
            epochs = [h['epoch'] for h in history]
            axes[0].plot(epochs, [h['train_loss'] for h in history], alpha=0.7, label=f"Fold {r['fold']}")
            axes[1].plot(epochs, [h['validation_log_loss'] for h in history], alpha=0.7)
        axes[0].set(xlabel='Observed epoch', ylabel='Weighted window CE', title=f'Inner training | seed {seed}')
        axes[1].set(xlabel='Observed epoch', ylabel='Unweighted participant log loss', title='Inner validation (selected candidate)')
        axes[0].legend(ncol=2, fontsize=8)
        fig.tight_layout()
        plt.show()
else:
    print('Complete all folds before displaying outer-test plots.')

## Optional export
Set `SAVE_RESULTS = True` to explicitly write a new ZIP to the Kaggle working directory. It contains configuration, implementation versions/hashes, input audit, fold/window manifests, inner trial histories/selection, normalization statistics and outer predictions/metrics. It contains no raw signals or model weights. Existing files are never overwritten. Partial exports retain `complete = false` and are not a full CV result or resumable training checkpoint.

Downloads are manual through Kaggle's output browser. For a publication result, preserve this bundle and the executed notebook, including environment and actual backend-check output.

In [ ]:
SAVE_RESULTS = False
EXPORT_PATH = Path('/kaggle/working/compact_mamba_cv_results.zip')
if SAVE_RESULTS:
    if 'results' not in globals() or not results.get('runs'):
        raise RuntimeError('No completed-fold results available to export')
    print('Saved:', cv.export_results(results, EXPORT_PATH))
else:
    print('No result files written. Results remain in RAM.')